# Benchmarking with ioh

This notebook sets up a benchmark experiment: several DE variants on the
[BBOB](https://numbbo.github.io/coco/testsuites/bbob) functions of
[IOHexperimenter](https://iohprofiler.github.io/) (ioh), with output that
[IOHanalyzer](https://iohanalyzer.liacs.nl) can load, including modde's
internal state.

The experiment here is deliberately tiny so that the documentation builds
quickly. Increase the functions, dimensions and repetitions for real use.

In [ ]:
import tempfile, pathlib
import numpy as np
import ioh
import modde

## A quick comparison in a loop

Any ioh problem can be passed to `modde.ModularDE`; bounds and the final target
come from the problem. Here, three variants on three BBOB functions, 5 runs
each, with the median precision $f - f^*$ reached:

In [ ]:
variants = {"DE": {}, "SHADE": {"preset": "SHADE"}, "L-SHADE": {"preset": "L-SHADE"}}
dim, budget, runs = 5, 5000, 5

print(f"{'function':<24}" + "".join(f"{name:>12}" for name in variants))
for fid in (1, 8, 15):
    row = []
    for name, settings in variants.items():
        precision = []
        for run in range(runs):
            problem = ioh.get_problem(fid, instance=run + 1, dimension=dim)
            modde.ModularDE(problem, budget=budget, seed=run, **settings).run()
            precision.append(problem.state.current_best.y - problem.optimum.y)
        row.append(np.median(precision))
    label = f"f{fid} {ioh.get_problem(fid, 1, dim).meta_data.name}"
    print(f"{label:<24}" + "".join(f"{v:12.2e}" for v in row))

Runs stop when the final target ($f^* + 10^{-8}$) is reached, so precisions
below $10^{-8}$ mean the function was solved.

## A full experiment with `ioh.Experiment`

`ioh.Experiment` runs an algorithm on every combination of functions,
instances and dimensions, and writes IOHanalyzer files. It expects an object
that can be **called with a problem**. For modde, a small class does it. It also
forwards the optimizer's state (`curr_F`, `curr_CR`, `corrected`, ...), so the
experiment can log it next to the objective values:

In [ ]:
class DEAlgorithm:
    """One modde run per problem; budget = budget_per_dim x dimension."""

    def __init__(self, budget_per_dim=2000, **settings):
        self.budget_per_dim, self.settings, self.opt = budget_per_dim, settings, None

    def __call__(self, problem):
        budget = self.budget_per_dim * problem.meta_data.n_variables
        self.opt = modde.ModularDE(problem, budget=budget, track_evaluations=True, **self.settings)
        self.opt.run()

    def __getattr__(self, name):
        # tracked attributes of the current run (0 before the first run)
        if name.startswith("_"):
            raise AttributeError(name)
        opt = self.__dict__.get("opt")
        return getattr(opt, name) if opt is not None else 0.0

`track_evaluations=True` makes the logged values belong to the individual just
evaluated (per-evaluation statistics). Now run two variants. Each gets its own
folder; together they form one IOHanalyzer data set:

In [ ]:
output = tempfile.mkdtemp()
for name, settings in {"DE": {}, "L-SHADE": {"preset": "L-SHADE"}}.items():
    experiment = ioh.Experiment(
        algorithm=DEAlgorithm(**settings),
        fids=[1, 8, 15], iids=[1, 2], dims=[5], reps=2,
        output_directory=output, folder_name=name, algorithm_name=name,
        logged_attributes=["curr_F", "curr_CR", "corrected"],
        zip_output=False,
    )
    experiment()

for path in sorted(pathlib.Path(output).rglob("*.json")):
    print(path.relative_to(output))

The data files contain the logged attributes as extra columns:

In [ ]:
data_file = next(pathlib.Path(output, "L-SHADE").rglob("*.dat"))
print("\n".join(data_file.read_text().splitlines()[:5]))

Zip the output folders (or keep ioh's default `zip_output=True`) and upload them to
[IOHanalyzer](https://iohanalyzer.liacs.nl) to compare the variants.

Things to change for a real experiment:

- `fids=range(1, 25)`, several dimensions, more instances and repetitions;
- `njobs=-1` to use all cores;
- `logger_triggers`: by default ioh logs on every improvement. Logging every
  evaluation (`ioh.logger.trigger.ALWAYS`) shows the full parameter dynamics but
  produces a lot of data;
- `experiment_attributes` to store, for example, the module settings with the data.